# 00 · Análisis y Mapeo de la Base de Datos — Savia Salud EPS

**Objetivo:** Entender la estructura de la BD MySQL de Savia Salud, mapear las relaciones entre tablas del módulo de Cuentas Médicas (`cm_*`) y establecer la base para construir el dataset de entrenamiento del modelo predictivo de glosas.

**Fuentes:** Archivos exportados de la BD en `data/Info_data_base/`
- `Estructura_columnas_llaves.csv` — Schema completo de las 554 tablas
- `ver_relaciones.csv` — Relaciones de llaves foráneas (FK)
- `Mostrar_primeras_2_filas_tabla.csv` — Queries `SELECT * LIMIT 2` por tabla

**Resultado esperado:** `data/Info_data_base/01_mapa_bd_facturas.md` + SQL de dataset

In [1]:
# Celda 1 — Importaciones y carga de archivos de metadata
import pandas as pd
from pathlib import Path

BASE      = Path(r"D:\Users\jcardonr\Documents\eafit\Codigo\data\Info_data_base")
REPO_ROOT = Path(r"D:\Users\jcardonr\Documents\eafit\Codigo")

# Cargar los tres archivos de metadata
df_estructura = pd.read_csv(BASE / "Estructura_columnas_llaves.csv")
df_relaciones = pd.read_csv(BASE / "ver_relaciones.csv")
df_queries    = pd.read_csv(BASE / "Mostrar_primeras_2_filas_tabla.csv")

print(f"Estructura BD:  {len(df_estructura):,} filas | {df_estructura['Tabla'].nunique():,} tablas únicas")
print(f"Relaciones FK:  {len(df_relaciones):,} relaciones")
print(f"Queries SELECT: {len(df_queries):,} queries (una por tabla)")
print("\nColumnas de Estructura_columnas_llaves.csv:", df_estructura.columns.tolist())
print("Columnas de ver_relaciones.csv:", df_relaciones.columns.tolist())

Estructura BD:  10,630 filas | 554 tablas únicas
Relaciones FK:  854 relaciones
Queries SELECT: 554 queries (una por tabla)

Columnas de Estructura_columnas_llaves.csv: ['Tabla', 'Columna', 'Tipo de Dato', 'Permite Nulos', 'Llave', 'Extra']
Columnas de ver_relaciones.csv: ['Tabla Origen', 'Columna Local', 'Tabla Destino', 'Columna Destino']


In [2]:
# Celda 2 — Resumen general: todas las tablas por prefijo
df_estructura.columns = df_estructura.columns.str.strip()
df_relaciones.columns = df_relaciones.columns.str.strip()

# Extraer prefijo de cada tabla (primeras 2-3 letras antes del primer '_')
df_estructura["prefijo"] = df_estructura["Tabla"].str.split("_").str[0]

resumen_prefijos = (
    df_estructura.groupby("prefijo")["Tabla"]
    .nunique()
    .sort_values(ascending=False)
    .reset_index()
    .rename(columns={"Tabla": "num_tablas"})
)
print("Tablas por módulo (prefijo):")
print(resumen_prefijos.to_string(index=False))

Tablas por módulo (prefijo):
prefijo  num_tablas
     cm          79
     au          70
   cntj          41
     gn          37
     mp          34
     ma          29
     pe          27
   aseg          23
    auc          23
    cnt          22
     tu          20
    gat          15
    aus          14
    rco          11
     gj          10
    ref          10
     ws          10
     gs           9
    inf           9
      v           8
     rt           8
    car           6
     aa           6
     cs           6
    ant           6
    mpc           4
    fin           3
     fe           2
     no           2
  siifa           1
    tmp           1
   temp           1
 flyway           1
     sc           1
archivo           1
  bases           1
   giro           1
    int           1
                  1


In [3]:
# Celda 3 — Tablas cm_ (Cuentas Médicas): estructura completa

tablas_cm = df_estructura[df_estructura["Tabla"].str.startswith("cm_")].copy()

cols_por_tabla = (
    tablas_cm.groupby("Tabla")["Columna"]
    .count()
    .sort_values(ascending=False)
    .reset_index()
    .rename(columns={"Columna": "num_columnas"})
)

print(f"Total tablas cm_: {cols_por_tabla['Tabla'].nunique()}")
print("\nColumnas por tabla cm_ (ordenado por complejidad):")
print(cols_por_tabla.to_string(index=False))

Total tablas cm_: 79

Columnas por tabla cm_ (ordenado por complejidad):
                               Tabla  num_columnas
                         cm_facturas            68
                         cm_detalles            63
                   cm_fe_rips_cargas            63
        cm_rips_ah_hospitalizaciones            50
  cm_rips_carga_ah_hospitalizaciones            50
                cm_rips_au_urgencias            44
          cm_rips_carga_au_urgencias            44
          cm_rips_carga_ac_consultas            44
                cm_rips_ac_consultas            44
     cm_rips_carga_ap_procedimientos            42
           cm_rips_ap_procedimientos            42
            cm_rips_carga_ct_control            41
                  cm_rips_ct_control            41
           cm_rips_carga_us_usuarios            35
                 cm_rips_us_usuarios            35
                      cm_rips_cargas            34
           cm_rips_an_recien_nacidos            33
     cm_r

In [4]:
# Celda 4 — Schema detallado de las tablas CRÍTICAS para el modelo

TABLAS_CRITICAS = [
    "cm_facturas",
    "cm_detalles",
    "cm_auditoria_motivos_glosas",
    "cm_fe_rips_cargas",
    "cm_fe_rips_carga_contenidos",
    "cm_factura_estados",
    "cm_auditoria_devoluciones",
    "cm_glosa_respuestas",
    "aseg_afiliados",
    "cnt_contratos",
    "cnt_prestadores",
    "cnt_prestador_sedes",
]

for tabla in TABLAS_CRITICAS:
    schema = df_estructura[df_estructura["Tabla"] == tabla]
    if len(schema) == 0:
        print(f"\n⚠️  {tabla}: NO ENCONTRADA en metadata")
        continue
    print(f"\n{'='*60}")
    print(f"📋 {tabla} ({len(schema)} columnas)")
    print(f"{'='*60}")
    print(schema[["Columna", "Tipo de Dato", "Permite Nulos", "Llave"]].to_string(index=False))


📋 cm_facturas (68 columnas)
                      Columna  Tipo de Dato Permite Nulos Llave
                           id           int            NO   PRI
               gn_empresas_id           int            NO   MUL
                 cm_grupos_id           int            NO   MUL
           cnt_prestadores_id           int            NO   MUL
            cm_rips_cargas_id           int           YES   MUL
         gn_usuarios_lider_id           int            NO   MUL
       gn_usuarios_tecnico_id           int           YES   MUL
        gn_usuarios_medico_id           int           YES   MUL
      gn_usuarios_gestiona_id           int           YES   MUL
         cm_fe_rips_cargas_id           int           YES   MUL
         mae_tipo_contrato_id           int            NO   MUL
     mae_tipo_contrato_codigo    varchar(8)           YES   MUL
      mae_tipo_contrato_valor  varchar(128)           YES   MUL
                          nit   varchar(16)            NO   MUL
           

In [5]:
# Celda 5 — Relaciones FK de tablas cm_ con el resto del sistema

df_relaciones.columns = [c.strip() for c in df_relaciones.columns]

# Relaciones donde el origen es una tabla cm_
rel_cm_origen = df_relaciones[
    df_relaciones["Tabla Origen"].str.startswith("cm_")
].copy()

# Relaciones donde el destino es una tabla cm_
rel_cm_destino = df_relaciones[
    df_relaciones["Tabla Destino"].str.startswith("cm_")
].copy()

print(f"Relaciones donde cm_ es ORIGEN:  {len(rel_cm_origen)}")
print(f"Relaciones donde cm_ es DESTINO: {len(rel_cm_destino)}")
print(f"\nRelaciones desde tablas cm_ hacia otras tablas:")
print(rel_cm_origen.sort_values("Tabla Origen").to_string(index=False))

Relaciones donde cm_ es ORIGEN:  115
Relaciones donde cm_ es DESTINO: 100

Relaciones desde tablas cm_ hacia otras tablas:
                        Tabla Origen                       Columna Local                 Tabla Destino Columna Destino
               cm_auditoria_adjuntos                      cm_facturas_id                   cm_facturas              id
               cm_auditoria_adjuntos                      cm_detalles_id                   cm_detalles              id
         cm_auditoria_autorizaciones                       au_anexos4_id                    au_anexos4              id
         cm_auditoria_autorizaciones                      cm_detalles_id                   cm_detalles              id
         cm_auditoria_autorizaciones                      cm_facturas_id                   cm_facturas              id
      cm_auditoria_capita_descuentos                      cm_detalles_id                   cm_detalles              id
      cm_auditoria_capita_descuentos        

In [6]:
# Celda 6 — Relaciones específicas de las tablas críticas para el modelo

for tabla in ["cm_facturas", "cm_detalles", "cm_auditoria_motivos_glosas", "cm_fe_rips_cargas"]:
    sale = df_relaciones[df_relaciones["Tabla Origen"] == tabla]
    entra = df_relaciones[df_relaciones["Tabla Destino"] == tabla]
    print(f"\n{'─'*50}")
    print(f"🔗 {tabla}")
    if len(sale) > 0:
        print(f"  ↗ REFERENCIA A:")
        for _, r in sale.iterrows():
            print(f"     {r['Columna Local']} → {r['Tabla Destino']}.{r['Columna Destino']}")
    if len(entra) > 0:
        print(f"  ↙ ES REFERENCIADA POR:")
        for _, r in entra.iterrows():
            print(f"     {r['Tabla Origen']}.{r['Columna Local']}")


──────────────────────────────────────────────────
🔗 cm_facturas
  ↗ REFERENCIA A:
     cm_fe_rips_cargas_id → cm_fe_rips_cargas.id
     cm_grupos_id → cm_grupos.id
     cm_rips_cargas_id → cm_rips_cargas.id
     cnt_prestadores_id → cnt_prestadores.id
     gn_empresas_id → gn_empresas.id
     gn_usuarios_lider_id → gn_usuarios.id
     gn_usuarios_medico_id → gn_usuarios.id
     gn_usuarios_tecnico_id → gn_usuarios.id
     gn_usuarios_gestiona_id → gn_usuarios.id
  ↙ ES REFERENCIADA POR:
     cm_auditoria_adjuntos.cm_facturas_id
     cm_auditoria_autorizaciones.cm_facturas_id
     cm_auditoria_cierres.cm_facturas_id
     cm_auditoria_devoluciones.cm_facturas_id
     cm_detalles.cm_facturas_id
     cm_factura_estados.cm_facturas_id
     cm_factura_transacciones.cm_facturas_id
     cm_fe_notas.cm_facturas_id
     cm_fe_rips_facturas.cm_facturas_id
     cm_fe_soportes.cm_facturas_id
     cm_glosa_respuestas.cm_facturas_id
     cm_historico_facturas.cm_facturas_id
     cm_pago_facturas.cm

In [7]:
# Celda 7 — Identificar columnas candidatas para la variable TARGET (estado de la factura)

# Buscar columnas que sugieran estado/resultado de la factura
palabras_target = ["estado", "glosa", "devolucion", "devoluc", "resultado", "aprobad", "rechaz", "pago", "pagad"]

print("Columnas candidatas para TARGET (estado final de la factura):")
for palabra in palabras_target:
    candidatas = df_estructura[
        df_estructura["Columna"].str.contains(palabra, case=False, na=False) &
        df_estructura["Tabla"].str.startswith("cm_")
    ][["Tabla", "Columna", "Tipo de Dato"]]
    if len(candidatas) > 0:
        print(f"\n  [{palabra}]")
        print(candidatas.to_string(index=False))

Columnas candidatas para TARGET (estado final de la factura):

  [estado]
                             Tabla                  Columna Tipo de Dato
       cm_auditoria_autorizaciones         nombre_prestador varchar(264)
               cm_auditoria_masiva       cnt_prestadores_id          int
               cm_auditoria_masiva           estado_proceso          int
                 cm_conciliaciones       cnt_prestadores_id          int
                 cm_conciliaciones           estado_proceso          int
                       cm_detalles                   estado          int
              cm_devolucion_masiva       cnt_prestadores_id          int
              cm_devolucion_masiva           estado_proceso          int
                cm_factura_estados           estado_factura          int
          cm_factura_transacciones                   estado          int
                       cm_facturas       cnt_prestadores_id          int
                       cm_facturas           estad

In [8]:
# Celda 8 — Extraer queries SELECT LIMIT 2 para tablas cm_ críticas

# Las tablas críticas cuyas primeras 2 filas queremos ver
tablas_explorar = [
    "cm_facturas", "cm_detalles", "cm_auditoria_motivos_glosas",
    "cm_fe_rips_cargas", "cm_fe_rips_carga_contenidos",
    "cm_factura_estados", "cm_auditoria_devoluciones",
    "aseg_afiliados", "cnt_contratos", "cnt_prestadores",
]

# Extraer queries del CSV
col_query = df_queries.columns[0]  # columna 'consultas_a_ejecutar'
queries_filtrados = df_queries[
    df_queries[col_query].str.contains(
        "|".join(tablas_explorar), case=False, na=False
    )
]

print(f"Queries encontrados para tablas críticas: {len(queries_filtrados)}")

# Guardar queries filtrados
output_sql = BASE / "queries_exploracion_cm.sql"
with open(output_sql, "w", encoding="utf-8") as f:
    f.write("-- Queries para explorar primeras 2 filas de tablas críticas\n")
    f.write("-- Proyecto: Analítica Predictiva Savia Salud EPS\n")
    f.write("-- Ejecutar contra BD: system_savia\n\n")
    for _, row in queries_filtrados.iterrows():
        f.write(row[col_query] + "\n")

print(f"\nQueries guardados en: {output_sql}")
print("\nContenido:")
for _, row in queries_filtrados.iterrows():
    print(" ", row[col_query])

Queries encontrados para tablas críticas: 15

Queries guardados en: D:\Users\jcardonr\Documents\eafit\Codigo\data\Info_data_base\queries_exploracion_cm.sql

Contenido:
  SELECT * FROM aa_mala_cm_fe_rips_cargas LIMIT 2;
  SELECT * FROM aa_temp_cm_fe_rips_cargas LIMIT 2;
  SELECT * FROM aseg_afiliados LIMIT 2;
  SELECT * FROM aseg_afiliados_ods LIMIT 2;
  SELECT * FROM cm_auditoria_devoluciones LIMIT 2;
  SELECT * FROM cm_auditoria_motivos_glosas LIMIT 2;
  SELECT * FROM cm_detalles LIMIT 2;
  SELECT * FROM cm_factura_estados LIMIT 2;
  SELECT * FROM cm_facturas LIMIT 2;
  SELECT * FROM cm_fe_rips_carga_contenidos LIMIT 2;
  SELECT * FROM cm_fe_rips_cargas LIMIT 2;
  SELECT * FROM cnt_contratos LIMIT 2;
  SELECT * FROM cnt_prestadores LIMIT 2;
  SELECT * FROM v_cnt_prestadores_axcel LIMIT 2;
  SELECT * FROM ws_cm_facturas LIMIT 2;


In [9]:
# Celda 9 — Generar documento 01_mapa_bd_facturas.md

def generar_schema_md(tabla: str, df: pd.DataFrame) -> str:
    """Genera tabla markdown con el schema de una tabla."""
    schema = df[df["Tabla"] == tabla]
    if len(schema) == 0:
        return f"\n> ⚠️ Tabla `{tabla}` no encontrada en metadata.\n"
    lineas = [f"\n### `{tabla}` ({len(schema)} columnas)\n"]
    lineas.append("| Columna | Tipo | Nulos | Llave |")
    lineas.append("|---|---|---|---|")
    for _, r in schema.iterrows():
        llave = r.get("Llave", "") or ""
        lineas.append(f"| `{r['Columna']}` | {r['Tipo de Dato']} | {r['Permite Nulos']} | {llave} |")
    return "\n".join(lineas)


def generar_relaciones_md(tabla: str, df_rel: pd.DataFrame) -> str:
    """Genera sección de relaciones FK para una tabla."""
    sale  = df_rel[df_rel["Tabla Origen"] == tabla]
    entra = df_rel[df_rel["Tabla Destino"] == tabla]
    lineas = []
    if len(sale) > 0:
        lineas.append("**Referencia hacia:**")
        for _, r in sale.iterrows():
            lineas.append(f"- `{r['Columna Local']}` → `{r['Tabla Destino']}.{r['Columna Destino']}`")
    if len(entra) > 0:
        lineas.append("\n**Es referenciada por:**")
        for _, r in entra.iterrows():
            lineas.append(f"- `{r['Tabla Origen']}.{r['Columna Local']}`")
    return "\n".join(lineas) if lineas else "_Sin relaciones FK encontradas en metadata._"


# Construir el documento
doc_lines = [
    "# Mapa de Base de Datos — Cuentas Médicas Savia Salud EPS",
    f"",
    f"**Generado:** 2026-03-04 | **Fuente:** Metadata BD MySQL `system_savia`",
    f"**Tablas totales:** {df_estructura['Tabla'].nunique()} | **Relaciones FK:** {len(df_relaciones)}",
    "",
    "---",
    "",
    "## 1. Flujo Completo de una Factura Médica",
    "",
    "```",
    "IPS (prestador de servicios)",
    "    │",
    "    ▼",
    "[cm_fe_rips_cargas]         ←── Radicación electrónica (FEV + RIPS)",
    "    │  cufe, cuv, factura_numero, factura_valor",
    "    │  Validaciones DIAN: de1601, de4401, de5001...",
    "    │",
    "    ├──[cm_fe_rips_carga_contenidos]  ←── Almacenamiento JSON/XML",
    "    │       tipo='json' → RIPS Res.2275  |  tipo='xml' → FEV DIAN",
    "    │",
    "    ▼",
    "[cm_facturas]               ←── Cabecera en el sistema",
    "    │  numero_radicado, valor_factura, estado_factura",
    "    │  fecha_radicacion, cnt_prestadores_id, gn_empresas_id",
    "    │",
    "    ▼",
    "[cm_detalles]               ←── Líneas de servicio (CUPS / CIE-10)",
    "    │  ma_servicio_id, valor_facturado, valor_pagado, valor_glosa",
    "    │  codigo_dx, aseg_afiliados_id",
    "    │  aplica_glosa (flag), aplica_autorizacion",
    "    │",
    "    ├── SI hay glosa:",
    "    │   [cm_auditoria_motivos_glosas]  ←── POR QUÉ se objetó",
    "    │       mae_motivo_id, mae_motivo_especifico_id",
    "    │       valor_motivo, tipologia",
    "    │",
    "    └── SI hay devolución:",
    "        [cm_auditoria_devoluciones]   ←── Error de forma",
    "            mae_devolucion_id, observacion",
    "",
    "ESTADO FINAL (TARGET del modelo predictivo):",
    "    0 = Auditada  → cm_detalles sin cm_auditoria_motivos_glosas",
    "    1 = Glosada   → cm_auditoria_motivos_glosas tiene registros",
    "    2 = Devuelta  → cm_auditoria_devoluciones tiene registros",
    "```",
    "",
    "---",
    "",
    "## 2. Módulos del Sistema (por prefijo de tabla)",
    "",
    "| Prefijo | Módulo | # Tablas |",
    "|---|---|---|",
]

# Tabla de prefijos
MODULOS = {
    "cm": "Cuentas Médicas (facturas, glosas, auditoría)",
    "aseg": "Asegurados / Afiliados",
    "cnt": "Contratos y Prestadores",
    "au": "Autorizaciones de Servicios",
    "auc": "Auditoría Clínica",
    "aus": "Auditoría de Solicitudes",
    "ma": "Catálogos Clínicos (CUPS, CIE-10, Medicamentos)",
    "mae": "Maestros (tipos, motivos, modalidades)",
    "gn": "Generales (usuarios, empresas, ubicaciones)",
    "pe": "Programas de Salud",
    "ant": "Anticipos",
    "car": "Caratulación",
    "ref": "Remisiones",
    "aa": "Auxiliares / Temporales",
}

for _, row in resumen_prefijos.iterrows():
    modulo = MODULOS.get(row["prefijo"], "Otros")
    doc_lines.append(f"| `{row['prefijo']}_*` | {modulo} | {row['num_tablas']} |")

doc_lines += [
    "",
    "---",
    "",
    "## 3. Schema de Tablas Críticas para el Modelo Predictivo",
    "",
]

# Agregar schema de cada tabla crítica
for tabla in TABLAS_CRITICAS:
    doc_lines.append(generar_schema_md(tabla, df_estructura))
    doc_lines.append("")
    doc_lines.append("**Relaciones FK:**")
    doc_lines.append(generar_relaciones_md(tabla, df_relaciones))
    doc_lines.append("")

doc_lines += [
    "---",
    "",
    "## 4. Todas las Relaciones entre Tablas cm_",
    "",
    "| Tabla Origen | Columna FK | Tabla Destino | Columna Ref |",
    "|---|---|---|---|",
]

for _, r in rel_cm_origen.sort_values("Tabla Origen").iterrows():
    doc_lines.append(f"| `{r['Tabla Origen']}` | `{r['Columna Local']}` | `{r['Tabla Destino']}` | `{r['Columna Destino']}` |")

doc_lines += [
    "",
    "---",
    "",
    "## 5. Columnas Candidatas para Variable TARGET",
    "",
    "Las siguientes columnas en tablas `cm_` son candidatas para determinar",
    "el estado final de la factura (Auditada=0 / Glosada=1 / Devuelta=2):",
    "",
    "| Tabla | Columna | Tipo |",
    "|---|---|---|",
]

for palabra in ["estado", "glosa", "devolucion", "devoluc", "resultado"]:
    candidatas = df_estructura[
        df_estructura["Columna"].str.contains(palabra, case=False, na=False) &
        df_estructura["Tabla"].str.startswith("cm_")
    ]
    for _, r in candidatas.iterrows():
        doc_lines.append(f"| `{r['Tabla']}` | `{r['Columna']}` | {r['Tipo de Dato']} |")

doc_lines += [
    "",
    "---",
    "",
    "## 6. Propuesta de Features para el Modelo Predictivo",
    "",
    "### 6.1 Features de Pertinencia Clínica (desde cm_detalles + RIPS)",
    "| Feature | Fuente | Tipo | Justificación |",
    "|---|---|---|---|",
    "| `codigo_dx` / CIE-10 | cm_detalles | categorical | Diagnóstico principal |",
    "| `ma_servicio_id` / CUPS | cm_detalles | categorical | Procedimiento facturado |",
    "| `dx_sexo_inconsistente` | cm_detalles + aseg_afiliados | bool | VAL-004 Res.2284 |",
    "| `num_dx_secundarios` | cm_rips_* | int | Complejidad del caso |",
    "| `nivel_complejidad_cups` | ma_servicios_habilitacion | int | Nivel requerido |",
    "| `nivel_habilitacion_ips` | cnt_prestador_sedes | int | Nivel real de la IPS |",
    "| `complejidad_mismatch` | derivada | bool | CUPS vs nivel IPS (VAL-003) |",
    "| `tiene_anestesiologia` | cm_detalles | bool | Cirugía sin anestesiólogo (VAL-005) |",
    "| `cups_autorizado_contrato` | cnt_contrato_detalles | bool | Servicio cubierto (VAL-003) |",
    "",
    "### 6.2 Features de Comportamiento del Prestador",
    "| Feature | Fuente | Tipo | Justificación |",
    "|---|---|---|---|",
    "| `tasa_glosa_historica_nit` | cm_facturas (12m) | float | Historial del prestador |",
    "| `tasa_devolucion_historica_nit` | cm_facturas (12m) | float | Historial devoluciones |",
    "| `valor_promedio_factura_nit` | cm_facturas | float | Patrón de facturación |",
    "| `dias_promedio_radicacion_nit` | cm_facturas | float | Puntualidad del prestador |",
    "| `concentracion_cups_prestador` | cm_detalles | float | HHI de CUPS facturados |",
    "| `trimestre_facturacion` | cm_facturas | int | Estacionalidad |",
    "",
    "### 6.3 Features Financieras y Administrativas",
    "| Feature | Fuente | Tipo | Justificación |",
    "|---|---|---|---|",
    "| `valor_total_factura` | cm_facturas | float | Magnitud económica |",
    "| `num_items_factura` | cm_detalles | int | Complejidad de la factura |",
    "| `modalidad_pago_encoded` | cm_facturas / cnt_contratos | int | Evento=0, Capita=1, Paquete=2 |",
    "| `dias_hasta_radicacion` | cm_facturas | int | VAL-002: extemporaneidad |",
    "| `discrepancia_fiscal_abs` | cm_facturas + cm_fe_rips_cargas | float | VAL-001: JSON vs XML |",
    "| `porcentaje_mayor_item` | cm_detalles | float | Concentración de valor |",
    "",
    "---",
    "",
    f"*Documento generado automáticamente desde metadata BD — 2026-03-04*",
]

# Escribir el documento
output_md = BASE / "01_mapa_bd_facturas.md"
with open(output_md, "w", encoding="utf-8") as f:
    f.write("\n".join(doc_lines))

print(f"✅ Documento generado: {output_md}")
print(f"   Líneas: {len(doc_lines)}")

✅ Documento generado: D:\Users\jcardonr\Documents\eafit\Codigo\data\Info_data_base\01_mapa_bd_facturas.md
   Líneas: 446


In [10]:
# Celda 10 — Verificación final: resumen de lo generado
from pathlib import Path

archivos_generados = [
    BASE / "01_mapa_bd_facturas.md",
    BASE / "02_queries_dataset_entrenamiento.sql",
    BASE / "queries_exploracion_cm.sql",
]

print("📁 Archivos generados en Info_data_base/:")
for f in archivos_generados:
    if f.exists():
        size_kb = f.stat().st_size / 1024
        print(f"  ✅ {f.name} ({size_kb:.1f} KB)")
    else:
        print(f"  ❌ {f.name} — NO GENERADO (verificar celda correspondiente)")

📁 Archivos generados en Info_data_base/:
  ✅ 01_mapa_bd_facturas.md (61.3 KB)
  ✅ 02_queries_dataset_entrenamiento.sql (11.7 KB)
  ✅ queries_exploracion_cm.sql (0.8 KB)
